# BBBC031: training Stochastic Mixture (NB=3 e NB=7) su Colab

Questo notebook addestra **Extended Stochastic Mixture NCA** su **5 immagini** BBBC031 per due dimensioni di vicinato (3 e 7), adatto a Google Colab.

**Prima di eseguire:**
1. **GPU (consigliata):** Menu *Runtime → Cambia tipo di runtime* e seleziona **GPU**. Se dopo `pip install -r requirements.txt` vedi "Torch not compiled with CUDA", riavvia il runtime e **non** rieseguire la cella di pip install (usa il PyTorch già presente su Colab), oppure reinstalla torch con CUDA.
2. Carica il dataset BBBC031 (cartella `BBBC031_v1_dataset` con sottocartella `Images/`) e il CSV ground truth su Google Drive, oppure caricali nella sessione Colab.
3. Imposta sotto i path `DATASET_DIR` e `CSV_PATH` (e opzionalmente `REPO_URL` se il repo è su GitHub).

## 1. Setup: clone repo e dipendenze

In [ ]:
# Clone del repo (cambia REPO_URL con il tuo repo se necessario)
REPO_URL = "https://github.com/luigidaddario/MNCA.git"  # oppure il path del tuo fork

!git clone --depth 1 {REPO_URL} /content/MNCA
%cd /content/MNCA
!pip install -q -r requirements.txt

## 2. Path dei dati (Drive o upload)

Scegli **una** delle due opzioni: monta Drive e imposta i path, oppure usa i path dove hai caricato i file nella sessione.

In [ ]:
# Opzione A: monta Google Drive e imposta i path alla cartella BBBC031
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = "/content/drive/MyDrive/BBBC031_v1_dataset"   # cartella con Images/ e Masks/
CSV_PATH    = "/content/drive/MyDrive/BBBC031_v1_DatasetGroundTruth.csv"  # CSV ground truth (sep=";")

# Opzione B: se hai caricato i file in /content/ (es. zip estratto):
# DATASET_DIR = "/content/BBBC031_v1_dataset"
# CSV_PATH    = "/content/BBBC031_v1_DatasetGroundTruth.csv"

import os
assert os.path.isdir(DATASET_DIR), f"Dataset non trovato: {DATASET_DIR}"
assert os.path.isfile(CSV_PATH), f"CSV non trovato: {CSV_PATH}"
assert os.path.isdir(os.path.join(DATASET_DIR, "Images")), "Manca la sottocartella Images/"
print("Path dati OK.")

In [ ]:
# Scegliamo 5 immagini dal CSV (con file CELLMASK presente)
import pandas as pd
images_dir = os.path.join(DATASET_DIR, "Images")
df_gt = pd.read_csv(CSV_PATH, sep=";")
all_names = df_gt["ImageName"].unique()
# Solo immagini per cui esiste il file _CELLMASK.png
available = [n for n in all_names if os.path.isfile(os.path.join(images_dir, f"{n}_CELLMASK.png"))]
EXAMPLE_IMAGES = available[:5]
assert len(EXAMPLE_IMAGES) >= 5, f"Servono almeno 5 immagini con CELLMASK; trovate {len(EXAMPLE_IMAGES)}."
print("Training su 5 immagini:", EXAMPLE_IMAGES)

## 3. Training: Stochastic Mixture su **5 immagini** (NB=3 e NB=7)

In [ ]:
# Esecuzione in-process così la barra di avanzamento (tqdm) si vede nel notebook
import sys
import os
os.chdir("/content/MNCA")
if "/content/MNCA" not in sys.path:
    sys.path.insert(0, "/content/MNCA")

from experiments.train_bbbc031_mnca import main

for img_idx, example_image in enumerate(EXAMPLE_IMAGES):
    for nb in (3, 7):
        ckpt = f"models/bbbc031_stochastic_NB{nb}_img{img_idx}.pth"
        print(f"\n--- Immagine {img_idx+1}/5: {example_image} | NB={nb} ---")
        sys.argv = [
            "train_bbbc031_mnca.py",
            "--dataset_dir", DATASET_DIR,
            "--csv_path", CSV_PATH,
            "--example_image", example_image,
            "--stochastic", "--neighborhood_size", str(nb),
            "--checkpoint_path", ckpt,
            "--total_steps", "4000",
        ]
        main()

*(Il training NB=3 e NB=7 per tutte e 5 le immagini è eseguito nella sezione 3.)*

In [ ]:
# Training già eseguito nella cella sopra (5 immagini × NB=3 e NB=7).
pass

## 5. (Opzionale) Figure di confronto e download

Le figure vengono salvate in `thesis-latex/figs/bbbc031/` durante il training. Qui generiamo le figure del demo per NB=3 e NB=7 e le mostriamo.

In [ ]:
import subprocess
import shutil
os.chdir("/content/MNCA")
fig_dir = "thesis-latex/figs/bbbc031"
for img_idx, example_image in enumerate(EXAMPLE_IMAGES):
    for nb in (3, 7):
        ckpt = f"models/bbbc031_stochastic_NB{nb}_img{img_idx}.pth"
        subprocess.run([
            "python", "experiments/bbbc031_mnca_demo.py",
            "--dataset_dir", DATASET_DIR, "--csv_path", CSV_PATH,
            "--example_image", example_image,
            "--checkpoint", ckpt, "--neighborhood_size", str(nb), "--stochastic",
            "--out_dir", fig_dir, "--num_steps", "20",
        ], check=True)
        src = os.path.join(fig_dir, f"bbbc031_gt_vs_mnca_NB{nb}_stochastic.png")
        dst = os.path.join(fig_dir, f"bbbc031_gt_vs_mnca_NB{nb}_img{img_idx}.png")
        if os.path.isfile(src):
            shutil.copy(src, dst)

In [ ]:
from IPython.display import Image, display

fig_dir = "/content/MNCA/thesis-latex/figs/bbbc031"
import glob
for path in sorted(glob.glob(os.path.join(fig_dir, "bbbc031_gt_vs_mnca_*.png"))):
    name = os.path.basename(path)
    print(name)
    display(Image(path, width=500))

## 6. Video dell'evoluzione NCA

Genera un video che mostra l'evoluzione della maschera passo dopo passo (da seed alla predizione finale). Su Colab serve ffmpeg (di solito già presente); altrimenti viene salvato un GIF.

In [ ]:
# Video per la prima immagine, modello NB=3 (cambia esempio o checkpoint se vuoi)
import subprocess
os.chdir("/content/MNCA")
video_dir = "thesis-latex/figs/bbbc031"
example_img = EXAMPLE_IMAGES[0]
ckpt = "models/bbbc031_stochastic_NB3_img0.pth"
subprocess.run([
    "python", "experiments/bbbc031_mnca_demo.py",
    "--dataset_dir", DATASET_DIR, "--csv_path", CSV_PATH,
    "--example_image", example_img, "--checkpoint", ckpt,
    "--neighborhood_size", "3", "--stochastic", "--num_steps", "20",
    "--out_dir", video_dir, "--save_video",
    "--video_path", video_dir + "/bbbc031_evolution_NB3_img0.mp4",
], check=True)

from IPython.display import Video
video_path = "/content/MNCA/thesis-latex/figs/bbbc031/bbbc031_evolution_NB3_img0.mp4"
if os.path.isfile(video_path):
    display(Video(video_path, width=400))
else:
    gif_path = video_path.replace(".mp4", ".gif")
    if os.path.isfile(gif_path):
        display(Image(gif_path, width=400))
    else:
        print("File video non trovato. Esegui prima il training e il demo con --save_video.")

In [ ]:
# Scarica i checkpoint, le loss, le figure e il video (zip)
!cd /content/MNCA && zip -r /content/bbbc031_stochastic_outputs.zip models/bbbc031_stochastic_NB*.pth models/*_loss.npy thesis-latex/figs/bbbc031/*.png thesis-latex/figs/bbbc031/*.mp4 thesis-latex/figs/bbbc031/*.gif 2>/dev/null || true
from google.colab import files
files.download("/content/bbbc031_stochastic_outputs.zip")